# MRI Data Asian — Viewer

Slice-by-slice viewer for `MRI_data_asian`.  
Structure: `MRI_data/{subject}/{Thigh|Calf}/` with channels Water, Fat, In_phase, Opp_phase and masks.

**How to use:**
1. Run all cells.
2. Pick a subject, region, and modality.
3. Toggle overlays and drag the slice slider.

In [1]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox, Checkbox, HTML
from IPython.display import display

DATA_ROOT = pathlib.Path('MRI_data_asian/MRI_data')

In [2]:
# ── Discover subjects ─────────────────────────────────────────────────────────

MODALITIES = ['Water', 'Fat', 'In_phase', 'Opp_phase']
REGIONS    = ['Thigh', 'Calf']

subjects = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir())
print(f'{len(subjects)} subjects: {subjects}')

25 subjects: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25']


In [3]:
# ── Label colours (thigh muscles, from thigh_muscle_segmentation_labels.json) ─

THIGH_LABEL_COLORS = {
    1:  ('rectus_femoris',     np.array([  0, 255,   0]) / 255),
    2:  ('vastus_lateralis',   np.array([255,   0,   0]) / 255),
    3:  ('vastus_intermedius', np.array([  0, 139, 139]) / 255),
    4:  ('vastus_medialis',    np.array([255, 136,   0]) / 255),
    5:  ('sartorius',          np.array([  0,   0, 255]) / 255),
    6:  ('gracilis',           np.array([255, 255,   0]) / 255),
    7:  ('biceps_femoris',     np.array([255,   0, 255]) / 255),
    8:  ('semitendinosus',     np.array([210, 180, 140]) / 255),
    9:  ('semimembranosus',    np.array([190,  83,  83]) / 255),
    10: ('adductor_brevis',    np.array([106,  90, 205]) / 255),
    11: ('adductor_longus',    np.array([  0, 255, 255]) / 255),
    12: ('adductor_magnus',    np.array([255, 124, 128]) / 255),
    13: ('gluteus_maximus',    np.array([255, 228, 225]) / 255),
}


def muscle_overlay(seg_sl, region, alpha=0.5):
    """RGBA overlay for mask_muscles slice. Uses named colours for thigh."""
    rgba = np.zeros((*seg_sl.shape, 4), dtype=np.float32)
    labels = [l for l in np.unique(seg_sl) if l != 0]
    if not labels:
        return rgba
    if region == 'Thigh':
        for lbl in labels:
            color = THIGH_LABEL_COLORS.get(lbl)
            if color:
                rgba[seg_sl == lbl, :3] = color[1]
                rgba[seg_sl == lbl,  3] = alpha
            else:
                c = plt.colormaps['tab20'](lbl % 20)
                rgba[seg_sl == lbl, :3] = c[:3]
                rgba[seg_sl == lbl,  3] = alpha
    else:
        cmap = plt.colormaps['tab20'].resampled(max(max(labels) + 1, 2))
        for lbl in labels:
            c = cmap(lbl - 1)
            rgba[seg_sl == lbl, :3] = c[:3]
            rgba[seg_sl == lbl,  3] = alpha
    return rgba


def sat_overlay(mask_sl, alpha=0.35):
    """Yellow semi-transparent overlay for mask_whole_muscle_SAT."""
    rgba = np.zeros((*mask_sl.shape, 4), dtype=np.float32)
    rgba[mask_sl > 0] = [1.0, 0.9, 0.0, alpha]
    return rgba


def legend_patches(region, labels_present):
    patches = []
    if region == 'Thigh':
        for lbl in sorted(labels_present):
            info = THIGH_LABEL_COLORS.get(lbl)
            if info:
                patches.append(mpatches.Patch(color=info[1], label=f'{lbl}: {info[0]}'))
    return patches


def load_norm(path):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)


def load_mask(path):
    return sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.int32)


_cache = {}

def get_data(subject, region):
    key = (subject, region)
    if key not in _cache:
        d = DATA_ROOT / subject / region
        entry = {}
        for mod in MODALITIES:
            p = d / f'{mod}.nii.gz'
            entry[mod] = load_norm(p) if p.exists() else None
        for mask_name in ('mask_muscles', 'mask_whole_muscle_SAT'):
            p = d / f'{mask_name}.nii.gz'
            entry[mask_name] = load_mask(p) if p.exists() else None
        _cache[key] = entry
    return _cache[key]

In [4]:
# ── Widgets ───────────────────────────────────────────────────────────────────

subject_dd = Dropdown(
    options=subjects, value=subjects[0],
    description='Subject:',
    layout=widgets.Layout(width='200px'),
)
region_dd = Dropdown(
    options=REGIONS, value='Thigh',
    description='Region:',
    layout=widgets.Layout(width='180px'),
)
modality_dd = Dropdown(
    options=MODALITIES, value='Water',
    description='Modality:',
    layout=widgets.Layout(width='200px'),
)
slice_sl = IntSlider(
    min=0, max=1, step=1, value=0,
    description='Slice:',
    layout=widgets.Layout(width='600px'),
)
show_muscles_cb = Checkbox(
    value=False, description='Muscle labels',
    indent=False, layout=widgets.Layout(width='160px'),
)
show_sat_cb = Checkbox(
    value=False, description='SAT mask',
    indent=False, layout=widgets.Layout(width='130px'),
)
status_lbl = HTML(value='')
out = widgets.Output()


def render(subject, region, modality, slice_idx, show_muscles, show_sat):
    try:
        data = get_data(subject, region)
    except Exception as e:
        with out:
            out.clear_output(wait=True)
            print(f'Error loading data: {e}')
        return

    img = data.get(modality)
    if img is None:
        status_lbl.value = f'<i style="color:orange">{modality} not found for {subject}/{region}</i>'
        return
    status_lbl.value = ''

    sl = img[slice_idx]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(sl, cmap='gray', origin='lower')

    patches = []
    if show_muscles and data.get('mask_muscles') is not None:
        seg_sl = data['mask_muscles'][slice_idx]
        ax.imshow(muscle_overlay(seg_sl, region), origin='lower')
        labels_present = [l for l in np.unique(seg_sl) if l != 0]
        patches = legend_patches(region, labels_present)

    if show_sat and data.get('mask_whole_muscle_SAT') is not None:
        sat_sl = data['mask_whole_muscle_SAT'][slice_idx]
        ax.imshow(sat_overlay(sat_sl), origin='lower')
        patches.append(mpatches.Patch(color=[1.0, 0.9, 0.0], label='whole muscle + SAT'))

    if patches:
        ax.legend(
            handles=patches, loc='upper right',
            fontsize=7, framealpha=0.7,
            ncol=2 if len(patches) > 6 else 1,
        )

    ax.set_title(f'Subject {subject}  |  {region}  |  {modality}  |  slice {slice_idx}', fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def _rerender(*_):
    render(
        subject_dd.value, region_dd.value, modality_dd.value,
        slice_sl.value, show_muscles_cb.value, show_sat_cb.value,
    )


def _reload_and_render(change):
    data = get_data(subject_dd.value, region_dd.value)
    ref  = next((v for v in data.values() if isinstance(v, np.ndarray) and v.ndim == 3), None)
    if ref is not None:
        slice_sl.max   = ref.shape[0] - 1
        slice_sl.value = ref.shape[0] // 2
    _rerender()


subject_dd.observe(_reload_and_render, names='value')
region_dd.observe(_reload_and_render, names='value')
modality_dd.observe(_rerender, names='value')
slice_sl.observe(_rerender, names='value')
show_muscles_cb.observe(_rerender, names='value')
show_sat_cb.observe(_rerender, names='value')

# Initial render
_reload_and_render(None)

display(VBox([
    HBox([subject_dd, region_dd, modality_dd,
          show_muscles_cb, show_sat_cb]),
    slice_sl,
    status_lbl,
    out,
]))